# Controlled GELU UniversalXAS batch-size sweep

This notebook runs a controlled batch-size experiment for the FEFF UniversalXAS head. It uses fixed 64D features from `OMNIXAS_E2E_UNIVERSAL_RUN` and does not retrain the encoder.

The four candidates use batch sizes 32, 64, 128, and 192. The learning rate remains fixed to isolate batch size. Larger batches get fewer optimizer updates per epoch, and this notebook does not apply linear learning-rate scaling.

Selection uses mean validation eta across all eight FEFF datasets. Validation selects the candidate. Test metrics are then calculated for all candidates for reporting only.


## 1. Imports and fixed configuration

The source run comes from `OMNIXAS_E2E_UNIVERSAL_RUN`. If that variable is not set, the notebook uses `REPO_ROOT.parent/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42`.


In [1]:
from __future__ import annotations

from collections.abc import Iterator
from dataclasses import dataclass
from hashlib import sha256
import json
import math
import os
from pathlib import Path
from typing import Any
from IPython.display import display

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import DataLoader, Sampler, TensorDataset
from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.training import PlModule

torch.set_float32_matmul_precision("medium")
ELEMENTS = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu"]
FEFF_DATASETS = [f"{element}_FEFF" for element in ELEMENTS]
INPUT_DIM, OUTPUT_DIM = 64, 141
HIDDEN_DIMS = [500, 500, 550]
UNIVERSAL_SEED = 44
UNIVERSAL_DROPOUT = 0.10
UNIVERSAL_LR = 5e-4
BATCH_SIZES = [32, 64, 128, 192]
UNIVERSAL_SCHEDULER = "ReduceLROnPlateau"
UNIVERSAL_SCHEDULER_MODE = "min"
UNIVERSAL_SCHEDULER_FACTOR = 0.5
UNIVERSAL_BASE_SCHEDULER_PATIENCE = 8
UNIVERSAL_SCHEDULER_FREQUENCY = 2
UNIVERSAL_MIN_LR = 1e-6
UNIVERSAL_MAX_EPOCHS = 800
UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE = 60
CHECK_VAL_EVERY_N_EPOCH = 2
UNIVERSAL_MONITOR = "val_median_mse"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def locate_repo_root(start: Path) -> Path:
    configured = os.environ.get("OMNIXAS_REPO_ROOT")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if (candidate / "pyproject.toml").is_file() and (candidate / "omnixas").is_dir():
            return candidate
        raise FileNotFoundError(f"OMNIXAS_REPO_ROOT is not an OmniXAS repository: {candidate}")
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "omnixas").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the OmniXAS repository")


def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def json_value(value: Any) -> Any:
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_value(item) for item in value]
    return value


def fingerprint(value: Any) -> str:
    return sha256(json.dumps(json_value(value), sort_keys=True, separators=(",", ":")).encode()).hexdigest()


REPO_ROOT = locate_repo_root(Path.cwd().resolve())
configured_source = os.environ.get("OMNIXAS_E2E_UNIVERSAL_RUN")
SOURCE_RUN = Path(configured_source).expanduser().resolve() if configured_source else (REPO_ROOT.parent / "fulltrainingcopy072726" / "m3gnetAll8E2EUniversal" / "e2e_universal_seed42").resolve()
FEATURES_DIR = SOURCE_RUN / "features"
ML_DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
MATERIAL_IDS_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"
RUN_DIR = SOURCE_RUN / "sweep_balanced_gelu_universal_batch_sizes"
for required in (SOURCE_RUN, FEATURES_DIR, ML_DATA_DIR, MATERIAL_IDS_DIR):
    if not required.exists():
        raise FileNotFoundError(f"Missing required path: {required}")
print(f"Source run: {SOURCE_RUN}")
print(f"Output run: {RUN_DIR}")
print(f"Device: {DEVICE}")


Source run: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42
Output run: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/sweep_balanced_gelu_universal_batch_sizes
Device: cuda:0


## 2. Load and validate fixed exported data

The exported feature files supply `X`. Exported `y` must match the canonical targets in `tutorial_omnixas/ml_data` exactly, row by row. The checks verify `[N,64]` features, `[N,141]` targets, finite values, ID counts, duplicate IDs, nonnegative integer site suffixes, and material split isolation.


In [2]:
@dataclass(frozen=True)
class LoadedSplit:
    split: MLSplits
    ids: dict[str, list[str]]
    materials: dict[str, list[str]]


def read_matrix(path: Path) -> np.ndarray:
    return np.atleast_2d(np.loadtxt(path, dtype=np.float32))


def read_ids(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def load_dataset(dataset: str) -> LoadedSplit:
    parts: dict[str, MLData] = {}
    ids_by_split: dict[str, list[str]] = {}
    materials_by_split: dict[str, list[str]] = {}
    seen_ids: set[str] = set()
    for split_name in ("train", "val", "test"):
        x_path = FEATURES_DIR / f"{dataset}_{split_name}_X.txt"
        exported_y_path = FEATURES_DIR / f"{dataset}_{split_name}_y.txt"
        canonical_y_path = ML_DATA_DIR / f"{dataset}_{split_name}_y.txt"
        ids_path = MATERIAL_IDS_DIR / f"{dataset}_{split_name}.txt"
        for path in (x_path, exported_y_path, canonical_y_path, ids_path):
            if not path.is_file():
                raise FileNotFoundError(f"Missing {dataset} {split_name} file: {path}")

        X = read_matrix(x_path)
        exported_y = read_matrix(exported_y_path)
        canonical_y = read_matrix(canonical_y_path)
        if X.shape != (X.shape[0], INPUT_DIM) or exported_y.shape != (exported_y.shape[0], OUTPUT_DIM):
            raise ValueError(f"Invalid exported dimensions for {dataset} {split_name}: X={X.shape}, y={exported_y.shape}")
        if canonical_y.shape != exported_y.shape or X.shape[0] != exported_y.shape[0]:
            raise ValueError(f"Row or target dimensions do not match for {dataset} {split_name}")
        if not np.isfinite(X).all() or not np.isfinite(exported_y).all() or not np.isfinite(canonical_y).all():
            raise ValueError(f"Non-finite data in {dataset} {split_name}")
        if not np.array_equal(exported_y, canonical_y):
            raise ValueError(f"Exported targets do not exactly match tutorial_omnixas/ml_data for {dataset} {split_name}")

        ids = read_ids(ids_path)
        if len(ids) != X.shape[0]:
            raise ValueError(f"ID row count mismatch for {dataset} {split_name}: IDs={len(ids)}, rows={X.shape[0]}")
        if len(set(ids)) != len(ids):
            raise ValueError(f"Duplicate full material/site IDs in {ids_path}")
        overlap = seen_ids.intersection(ids)
        if overlap:
            raise ValueError(f"Duplicate full IDs across splits for {dataset}: {sorted(overlap)[:5]}")
        seen_ids.update(ids)
        materials = []
        for row in ids:
            material, separator, site = row.rpartition("_")
            if not separator or not material or not site or not site.isascii() or not site.isdigit() or int(site) < 0:
                raise ValueError(f"Invalid material/site ID with nonnegative integer site suffix in {ids_path}: {row!r}")
            materials.append(material)
        ids_by_split[split_name] = ids
        materials_by_split[split_name] = materials
        parts[split_name] = MLData(X=X, y=exported_y)

    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = set(materials_by_split[left]).intersection(materials_by_split[right])
        if overlap:
            raise ValueError(f"Material split isolation failed for {dataset} {left}/{right}: {sorted(overlap)[:5]}")
    return LoadedSplit(MLSplits(**parts), ids_by_split, materials_by_split)


loaded = {dataset: load_dataset(dataset) for dataset in FEFF_DATASETS}
splits = {dataset: item.split for dataset, item in loaded.items()}
source_files = []
for dataset in FEFF_DATASETS:
    for split_name in ("train", "val", "test"):
        source_files.extend(
            [
                FEATURES_DIR / f"{dataset}_{split_name}_X.txt",
                FEATURES_DIR / f"{dataset}_{split_name}_y.txt",
                ML_DATA_DIR / f"{dataset}_{split_name}_y.txt",
                MATERIAL_IDS_DIR / f"{dataset}_{split_name}.txt",
            ]
        )
source_files = sorted(set(source_files))
print({dataset: {name: len(getattr(split, name)) for name in ("train", "val", "test")} for dataset, split in splits.items()})


{'Ti_FEFF': {'train': 5140, 'val': 641, 'test': 641}, 'V_FEFF': {'train': 8653, 'val': 1080, 'test': 1080}, 'Cr_FEFF': {'train': 2457, 'val': 305, 'test': 305}, 'Mn_FEFF': {'train': 13644, 'val': 1704, 'test': 1704}, 'Fe_FEFF': {'train': 9657, 'val': 1205, 'test': 1205}, 'Co_FEFF': {'train': 8605, 'val': 1074, 'test': 1074}, 'Ni_FEFF': {'train': 3471, 'val': 432, 'test': 432}, 'Cu_FEFF': {'train': 3340, 'val': 416, 'test': 416}}


## 3. GELU model, exact balanced sampler, and budget self-check

The local model is `64 -> 500 -> 500 -> 550 -> 141`. Each hidden layer uses `Linear -> BatchNorm1d -> GELU -> Dropout`. The final layer uses `Softplus`.

`ElementStratifiedBatchSampler` redraws without replacement from each element pool at every epoch, using `seed + epoch`. A separate self-check sampler is created for every candidate before any real training sampler is created.


In [3]:
class GELUXASBlock(nn.Sequential):
    def __init__(self, input_dim: int, hidden_dims: list[int], output_dim: int, dropout: float):
        dims = [input_dim, *hidden_dims, output_dim]
        layers: list[nn.Module] = []
        for index, (width_in, width_out) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(width_in, width_out))
            if index < len(dims) - 2:
                layers.extend([nn.BatchNorm1d(width_out), nn.GELU(), nn.Dropout(dropout)])
            else:
                layers.append(nn.Softplus())
        super().__init__(*layers)


class ElementStratifiedBatchSampler(Sampler[list[int]]):
    def __init__(self, indices_by_element: dict[int, list[int]], per_element: int, seed: int):
        if not indices_by_element or set(indices_by_element) != set(range(len(ELEMENTS))):
            raise ValueError("Sampler requires one non-empty pool for each of the eight elements")
        if per_element < 1:
            raise ValueError("per_element must be positive")
        self.indices_by_element = {int(key): list(value) for key, value in indices_by_element.items()}
        self.element_order = sorted(self.indices_by_element)
        self.per_element = int(per_element)
        self.seed = int(seed)
        self.epoch = 0
        counts = [len(self.indices_by_element[key]) for key in self.element_order]
        self.n_batches = min(counts) // self.per_element
        if self.n_batches < 1:
            raise ValueError("Every element needs at least per_element rows")

    def __len__(self) -> int:
        return self.n_batches

    def __iter__(self) -> Iterator[list[int]]:
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        selected = {
            element: rng.permutation(self.indices_by_element[element])[: self.n_batches * self.per_element]
            for element in self.element_order
        }
        batches = []
        for batch_number in range(self.n_batches):
            batch = np.concatenate(
                [
                    selected[element][batch_number * self.per_element : (batch_number + 1) * self.per_element]
                    for element in self.element_order
                ]
            )
            rng.shuffle(batch)
            batches.append(batch.tolist())
        return iter(batches)


def combine_split(split_name: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    X_blocks, y_blocks, element_blocks = [], [], []
    for element_index, dataset in enumerate(FEFF_DATASETS):
        part = getattr(splits[dataset], split_name)
        X_blocks.append(part.X)
        y_blocks.append(part.y)
        element_blocks.append(np.full(len(part), element_index, dtype=np.int64))
    return np.concatenate(X_blocks), np.concatenate(y_blocks), np.concatenate(element_blocks)


universal_train_X, universal_train_y, train_element_codes = combine_split("train")
universal_val_X, universal_val_y, val_element_codes = combine_split("val")
train_indices_by_element = {i: np.flatnonzero(train_element_codes == i).tolist() for i in range(len(ELEMENTS))}
val_baselines = np.array([np.median(np.mean((splits[d].val.y - splits[d].train.y.mean(axis=0)) ** 2, axis=1)) for d in FEFF_DATASETS], dtype=np.float32)

candidate_specs = []
for batch_size in BATCH_SIZES:
    if batch_size % len(ELEMENTS):
        raise ValueError(f"Batch size must divide evenly across elements: {batch_size}")
    candidate_specs.append({"batch_size": batch_size, "rows_per_element": batch_size // len(ELEMENTS)})

sampler_checks = []
for candidate in candidate_specs:
    batch_size, per_element = candidate["batch_size"], candidate["rows_per_element"]
    check_sampler = ElementStratifiedBatchSampler(train_indices_by_element, per_element, UNIVERSAL_SEED)
    expected_batches = min(map(len, train_indices_by_element.values())) // per_element
    check_batches = list(check_sampler)
    if len(check_batches) != expected_batches:
        raise AssertionError(f"Expected {expected_batches} batches for {batch_size}, got {len(check_batches)}")
    sampled = []
    for batch in check_batches:
        if len(batch) != batch_size:
            raise AssertionError(f"Wrong batch size for {batch_size}: {len(batch)}")
        indices = np.asarray(batch, dtype=np.int64)
        if np.any(indices < 0) or np.any(indices >= len(universal_train_y)):
            raise AssertionError(f"Out-of-range sampled index for {batch_size}")
        counts = np.bincount(train_element_codes[indices], minlength=len(ELEMENTS))
        if not np.array_equal(counts, np.full(len(ELEMENTS), per_element)):
            raise AssertionError(f"Unequal element counts for {batch_size}: {counts.tolist()}")
        sampled.extend(batch)
    if len(sampled) != len(set(sampled)):
        raise AssertionError(f"Duplicate row index in one epoch for {batch_size}")
    full_batches = math.ceil(len(universal_train_y) / batch_size)
    balanced_batches = len(check_sampler)
    scale = full_batches / balanced_batches
    candidate.update({"full_data_batches_per_epoch": full_batches, "balanced_batches_per_epoch": balanced_batches, "update_budget_scale": scale, "effective_scheduler_patience": round(UNIVERSAL_BASE_SCHEDULER_PATIENCE * scale), "effective_early_stopping_patience": round(UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE * scale)})
    sampler_checks.append({"batch_size": batch_size, "rows_per_element": per_element, "expected_batches": expected_batches})
print("Sampler self-checks passed")
print(pd.DataFrame(sampler_checks).to_string(index=False))
print(pd.DataFrame(candidate_specs).to_string(index=False))


Sampler self-checks passed
 batch_size  rows_per_element  expected_batches
         32                 4               614
         64                 8               307
        128                16               153
        192                24               102
 batch_size  rows_per_element  full_data_batches_per_epoch  balanced_batches_per_epoch  update_budget_scale  effective_scheduler_patience  effective_early_stopping_patience
         32                 4                         1718                         614             2.798046                            22                                168
         64                 8                          859                         307             2.798046                            22                                168
        128                16                          430                         153             2.810458                            22                                169
        192                24            

## 4. Matching run state and candidate settings

Reuse requires matching settings, a complete matching `TRAINING_COMPLETE.json`, exactly one best checkpoint, and a matching SHA256. Reject every incomplete non-empty candidate directory, even when it contains a best checkpoint.


In [4]:
candidate_settings = []
for candidate in candidate_specs:
    candidate_settings.append({
        "model": "UniversalXAS", "activation": "GELU", "architecture": "64 -> 500 -> 500 -> 550 -> 141",
        "input_dim": INPUT_DIM, "hidden_dims": HIDDEN_DIMS, "output_dim": OUTPUT_DIM, "seed": UNIVERSAL_SEED,
        "dropout": UNIVERSAL_DROPOUT, "optimizer": "Adam", "lr": UNIVERSAL_LR,
        "batch_size": candidate["batch_size"], "rows_per_element": candidate["rows_per_element"],
        "sampler": "ElementStratifiedBatchSampler", "sampler_redraw": "without replacement per element pool with seed + epoch",
        "scheduler": UNIVERSAL_SCHEDULER, "scheduler_mode": UNIVERSAL_SCHEDULER_MODE, "scheduler_factor": UNIVERSAL_SCHEDULER_FACTOR,
        "scheduler_base_patience": UNIVERSAL_BASE_SCHEDULER_PATIENCE, "scheduler_effective_patience": candidate["effective_scheduler_patience"],
        "scheduler_frequency": UNIVERSAL_SCHEDULER_FREQUENCY, "min_lr": UNIVERSAL_MIN_LR, "max_epochs": UNIVERSAL_MAX_EPOCHS,
        "early_stopping_base_patience": UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE, "early_stopping_effective_patience": candidate["effective_early_stopping_patience"],
        "validation_every_n_epochs": CHECK_VAL_EVERY_N_EPOCH, "monitor": UNIVERSAL_MONITOR,
        "full_data_batches_per_epoch": candidate["full_data_batches_per_epoch"], "balanced_batches_per_epoch": candidate["balanced_batches_per_epoch"], "update_budget_scale": candidate["update_budget_scale"],
        "source_run": str(SOURCE_RUN), "feature_provenance": "fixed exported 64D FEFF features from OMNIXAS_E2E_UNIVERSAL_RUN", "encoder_retrained": False,
    })
settings_core = {
    "experiment": "controlled batch-size experiment", "source_run": str(SOURCE_RUN),
    "feature_provenance": "fixed exported 64D FEFF features from OMNIXAS_E2E_UNIVERSAL_RUN", "encoder_retrained": False,
    "architecture": {"input_dim": INPUT_DIM, "hidden_dims": HIDDEN_DIMS, "output_dim": OUTPUT_DIM, "hidden_block": "Linear -> BatchNorm1d -> GELU -> Dropout", "output_block": "Linear -> Softplus"},
    "candidate_settings": candidate_settings,
    "selection_policy": "Select the unique candidate with maximum mean validation eta across all eight FEFF datasets. Do not use test metrics for selection.",
    "learning_rate_policy": "Keep Adam learning rate fixed at 5e-4. Do not apply linear learning-rate scaling.",
}
settings_fingerprint = fingerprint(settings_core)
manifest_path = RUN_DIR / "settings_manifest.json"
if RUN_DIR.exists():
    if not RUN_DIR.is_dir():
        raise FileExistsError(f"Output path is not a directory: {RUN_DIR}")
    if any(RUN_DIR.iterdir()):
        if not manifest_path.is_file():
            raise RuntimeError(f"Non-empty run directory has no settings_manifest.json: {RUN_DIR}")
        existing = json.loads(manifest_path.read_text(encoding="utf-8"))
        if existing.get("configuration_fingerprint") != settings_fingerprint:
            raise RuntimeError("Existing run settings do not match this notebook")
else:
    RUN_DIR.mkdir(parents=True)
manifest = {**settings_core, "configuration_fingerprint": settings_fingerprint, "status": "incomplete", "output_run": str(RUN_DIR), "source_files": [{"path": str(p), "sha256": sha256_file(p)} for p in source_files], "checkpoint_hashes_recorded": False}
manifest_path.write_text(json.dumps(json_value(manifest), indent=2), encoding="utf-8")


27185

## 5. Training and checkpoint helpers

The trainer uses progress bars, `CSVLogger`, validation every two epochs, `ReduceLROnPlateau`, and `val_median_mse` checkpoint monitoring. Each candidate uses its computed update-budget patience values.


In [5]:
class UniversalBalancedData(pl.LightningDataModule):
    def __init__(self, sampler: ElementStratifiedBatchSampler):
        super().__init__()
        self.train = TensorDataset(
            torch.as_tensor(universal_train_X, dtype=torch.float32),
            torch.as_tensor(universal_train_y, dtype=torch.float32),
            torch.as_tensor(train_element_codes, dtype=torch.long),
        )
        self.val = TensorDataset(
            torch.as_tensor(universal_val_X, dtype=torch.float32),
            torch.as_tensor(universal_val_y, dtype=torch.float32),
            torch.as_tensor(val_element_codes, dtype=torch.long),
        )
        self.sampler = sampler

    def train_dataloader(self):
        return DataLoader(self.train, batch_sampler=self.sampler)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=1024, shuffle=False)


class StandardTaskData(pl.LightningDataModule):
    def __init__(self, split: MLSplits, batch_size: int):
        super().__init__()
        self.train = TensorDataset(torch.as_tensor(split.train.X), torch.as_tensor(split.train.y))
        self.val = TensorDataset(torch.as_tensor(split.val.X), torch.as_tensor(split.val.y))
        self.batch_size = batch_size

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=1024, shuffle=False)


class BalancedUniversalModule(PlModule):
    def __init__(self, val_baselines: np.ndarray, **kwargs):
        super().__init__(**kwargs)
        self.register_buffer("val_baselines", torch.as_tensor(val_baselines, dtype=torch.float32))
        self.val_mses: list[torch.Tensor] = []
        self.val_elements: list[torch.Tensor] = []

    def training_step(self, batch, batch_idx):
        x, y, _ = batch
        return self.logged_loss("train_loss", y, self.model(x))

    def on_validation_epoch_start(self):
        self.val_mses, self.val_elements = [], []

    def validation_step(self, batch, batch_idx):
        x, y, elements = batch
        y_pred = self.model(x)
        self.val_mses.append(torch.mean((y - y_pred) ** 2, dim=1).detach())
        self.val_elements.append(elements.detach())
        return self.logged_loss("val_loss", y, y_pred)

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        if not self.val_mses:
            raise RuntimeError("Validation produced no rows")
        mses = torch.cat(self.val_mses)
        elements = torch.cat(self.val_elements)
        self.log("val_median_mse", torch.quantile(mses, 0.5), on_step=False, on_epoch=True, prog_bar=True)
        relative_medians = []
        for element_index, element in enumerate(ELEMENTS):
            mask = elements == element_index
            if not mask.any():
                raise RuntimeError(f"Validation has no rows for {element}")
            relative_medians.append(torch.quantile(mses[mask], 0.5) / self.val_baselines[element_index])
        self.log("val_balanced_rel_mse", torch.stack(relative_medians).mean(), on_step=False, on_epoch=True, prog_bar=True)


TRAINING_COMPLETE_FILENAME = "TRAINING_COMPLETE.json"


def _validate_setting_dir(setting_dir: Path) -> Path:
    setting_dir = setting_dir.resolve()
    run_root = RUN_DIR.resolve()
    if not setting_dir.is_relative_to(run_root):
        raise ValueError(f"Setting directory is outside output run: {setting_dir}")
    if setting_dir.exists() and not setting_dir.is_dir():
        raise FileExistsError(f"Setting path is not a directory: {setting_dir}")
    return setting_dir


def _write_setting_settings(setting_dir: Path, settings: dict[str, Any]) -> str:
    settings_fingerprint = fingerprint(settings)
    (setting_dir / "settings.json").write_text(
        json.dumps({"settings": json_value(settings), "fingerprint": settings_fingerprint}, indent=2),
        encoding="utf-8",
    )
    return settings_fingerprint


def _validate_setting_settings(setting_dir: Path, settings: dict[str, Any]) -> str:
    settings_path = setting_dir / "settings.json"
    if not settings_path.is_file():
        raise RuntimeError(f"Non-empty setting directory has no settings.json or completion marker: {setting_dir}")
    try:
        saved = json.loads(settings_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise RuntimeError(f"Cannot read setting settings: {settings_path}") from error
    settings_fingerprint = fingerprint(settings)
    if (
        not isinstance(saved, dict)
        or saved.get("fingerprint") != settings_fingerprint
        or fingerprint(saved.get("settings")) != settings_fingerprint
    ):
        raise RuntimeError(f"Checkpoint settings mismatch: {setting_dir}")
    return settings_fingerprint


def _best_candidates(setting_dir: Path) -> list[Path]:
    return sorted(path.resolve() for path in setting_dir.glob("best*.ckpt") if path.is_file())


def _validated_completion(setting_dir: Path, settings: dict[str, Any], checkpoint: Path) -> Path:
    marker_path = setting_dir / TRAINING_COMPLETE_FILENAME
    if not marker_path.is_file():
        raise RuntimeError(f"Non-empty setting directory has no valid {TRAINING_COMPLETE_FILENAME}: {setting_dir}")
    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise RuntimeError(f"Cannot read setting completion marker: {marker_path}") from error
    expected_fingerprint = fingerprint(settings)
    expected_checkpoint = checkpoint.resolve()
    expected_sha256 = sha256_file(expected_checkpoint)
    if (
        not isinstance(marker, dict)
        or marker.get("status") != "complete"
        or marker.get("settings_fingerprint") != expected_fingerprint
        or marker.get("checkpoint") != str(expected_checkpoint)
        or marker.get("checkpoint_sha256") != expected_sha256
    ):
        raise RuntimeError(f"Invalid or mismatched {TRAINING_COMPLETE_FILENAME}: {marker_path}")
    return expected_checkpoint


def best_checkpoint(setting_dir: Path, settings: dict[str, Any]) -> Path | None:
    setting_dir = _validate_setting_dir(setting_dir)
    if not setting_dir.exists():
        setting_dir.mkdir(parents=True)
        _write_setting_settings(setting_dir, settings)
        return None
    if not any(setting_dir.iterdir()):
        _write_setting_settings(setting_dir, settings)
        return None
    _validate_setting_settings(setting_dir, settings)
    if not (setting_dir / TRAINING_COMPLETE_FILENAME).is_file():
        raise RuntimeError(f"Non-empty setting directory has no valid {TRAINING_COMPLETE_FILENAME}: {setting_dir}")
    candidates = _best_candidates(setting_dir)
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one best*.ckpt before reuse in setting directory {setting_dir}, found {len(candidates)}")
    return _validated_completion(setting_dir, settings, candidates[0])


def finalize_training(setting_dir: Path, settings: dict[str, Any]) -> Path:
    setting_dir = _validate_setting_dir(setting_dir)
    if not setting_dir.exists():
        raise RuntimeError(f"Cannot finalize training in missing setting directory: {setting_dir}")
    settings_fingerprint = _validate_setting_settings(setting_dir, settings)
    candidates = _best_candidates(setting_dir)
    if len(candidates) != 1:
        raise RuntimeError(f"Training finished without exactly one best*.ckpt in {setting_dir}, found {len(candidates)}")
    checkpoint = candidates[0]
    marker = {
        "status": "complete",
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": sha256_file(checkpoint),
        "settings_fingerprint": settings_fingerprint,
    }
    (setting_dir / TRAINING_COMPLETE_FILENAME).write_text(json.dumps(marker, indent=2), encoding="utf-8")
    return checkpoint


def checkpoint_state(checkpoint: Path) -> dict[str, torch.Tensor]:
    payload = torch.load(checkpoint, map_location="cpu")
    state = payload.get("state_dict")
    if not state:
        raise ValueError(f"Checkpoint has no state_dict: {checkpoint}")
    result = {key.removeprefix("model."): value for key, value in state.items() if key.startswith("model.")}
    if not result:
        raise ValueError(f"Checkpoint has no model state: {checkpoint}")
    return result


def load_gelu_model(checkpoint: Path, dropout: float) -> GELUXASBlock:
    model = GELUXASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, dropout)
    model.load_state_dict(checkpoint_state(checkpoint), strict=True)
    return model.to(DEVICE).eval()


def make_callbacks(setting_dir: Path, monitor: str, patience: int, name: str):
    return [
        EarlyStopping(monitor=monitor, mode="min", patience=patience),
        ModelCheckpoint(
            dirpath=str(setting_dir),
            filename=f"best-{name}-{{epoch:03d}}-{{{monitor}:.8f}}",
            monitor=monitor,
            mode="min",
            save_top_k=1,
            save_last=True,
        ),
    ]


## 6. Train or safely reuse all four candidates

The real training sampler is created only after the separate self-checks. Every candidate keeps seed 44, dropout 0.10, Adam learning rate `5e-4`, and the fixed architecture.


In [6]:
candidate_checkpoints = {}
for candidate, settings in zip(candidate_specs, candidate_settings):
    batch_size = int(candidate["batch_size"])
    candidate_dir = RUN_DIR / f"batch_size_{batch_size}"
    checkpoint = best_checkpoint(candidate_dir, settings)
    if checkpoint is None:
        pl.seed_everything(UNIVERSAL_SEED, workers=True)
        real_sampler = ElementStratifiedBatchSampler(train_indices_by_element, int(candidate["rows_per_element"]), UNIVERSAL_SEED)
        module = BalancedUniversalModule(
            val_baselines=val_baselines,
            model=GELUXASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, UNIVERSAL_DROPOUT),
            optimizer=torch.optim.Adam, lr=UNIVERSAL_LR,
            lr_scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau,
            lr_scheduler_kwargs={"mode": UNIVERSAL_SCHEDULER_MODE, "factor": UNIVERSAL_SCHEDULER_FACTOR, "patience": int(candidate["effective_scheduler_patience"]), "min_lr": UNIVERSAL_MIN_LR},
            lr_scheduler_interval="epoch", lr_scheduler_frequency=UNIVERSAL_SCHEDULER_FREQUENCY, lr_scheduler_monitor=UNIVERSAL_MONITOR,
        )
        trainer = pl.Trainer(
            max_epochs=UNIVERSAL_MAX_EPOCHS, accelerator="auto", devices=1,
            check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
            callbacks=make_callbacks(candidate_dir, UNIVERSAL_MONITOR, int(candidate["effective_early_stopping_patience"]), f"batch{batch_size}"),
            logger=CSVLogger(save_dir=str(candidate_dir), name="logs"), enable_progress_bar=True, log_every_n_steps=1,
        )
        trainer.fit(module, datamodule=UniversalBalancedData(real_sampler))
        checkpoint = finalize_training(candidate_dir, settings)
    else:
        print(f"Reusing completed batch-size {batch_size} candidate: {checkpoint}")
    candidate_checkpoints[batch_size] = checkpoint
if set(candidate_checkpoints) != set(BATCH_SIZES):
    raise AssertionError("Expected one completed checkpoint for each batch size")


Reusing completed batch-size 32 candidate: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/sweep_balanced_gelu_universal_batch_sizes/batch_size_32/best-batch32-epoch=743-val_median_mse=0.00214526.ckpt
Reusing completed batch-size 64 candidate: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/sweep_balanced_gelu_universal_batch_sizes/batch_size_64/best-batch64-epoch=613-val_median_mse=0.00207035.ckpt
Reusing completed batch-size 128 candidate: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/sweep_balanced_gelu_universal_batch_sizes/batch_size_128/best-batch128-epoch=781-val_median_mse=0.00206784.ckpt
Reusing completed batch-size 192 candidate: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/sweep_balanced_gelu_universal_batch_sizes/batch_size_192/best-batch192-epoch=799-val_median_mse=0.00210384.ckpt


## 7. Validation-only evaluation and global candidate selection

Each candidate is evaluated on validation for each FEFF task. Eta uses that task's training-mean spectrum. The global selection score is mean validation eta across the eight tasks. No test metric is read or calculated in this section.


In [7]:
def predict(model: nn.Module, X: np.ndarray) -> np.ndarray:
    loader = DataLoader(TensorDataset(torch.as_tensor(X, dtype=torch.float32)), batch_size=1024, shuffle=False)
    values = []
    model.eval()
    with torch.inference_mode():
        for (batch,) in loader:
            values.append(model(batch.to(DEVICE)).cpu().numpy())
    return np.concatenate(values)


def validation_metrics(split: MLSplits, predictions: np.ndarray) -> dict[str, float]:
    target = split.val.y
    train_mean = split.train.y.mean(axis=0, keepdims=True)
    model_mse = float(np.median(np.mean((target - predictions) ** 2, axis=1)))
    baseline_mse = float(np.median(np.mean((target - train_mean) ** 2, axis=1)))
    if model_mse <= 0 or baseline_mse <= 0:
        raise ValueError(f"Invalid validation MSE: model={model_mse}, baseline={baseline_mse}")
    return {"val_median_mse": model_mse, "val_baseline_median_mse": baseline_mse, "val_eta": baseline_mse / model_mse}

candidate_validation_rows = []
for candidate in candidate_specs:
    batch_size = int(candidate["batch_size"])
    checkpoint = candidate_checkpoints[batch_size]
    model = load_gelu_model(checkpoint, UNIVERSAL_DROPOUT)
    for dataset in FEFF_DATASETS:
        row = {"batch_size": batch_size, "rows_per_element": int(candidate["rows_per_element"]), "dataset": dataset, "element": dataset.removesuffix("_FEFF"), "checkpoint": str(checkpoint), "checkpoint_sha256": sha256_file(checkpoint), "seed": UNIVERSAL_SEED, "dropout": UNIVERSAL_DROPOUT, "lr": UNIVERSAL_LR, "max_epochs": UNIVERSAL_MAX_EPOCHS, "effective_scheduler_patience": int(candidate["effective_scheduler_patience"]), "effective_early_stopping_patience": int(candidate["effective_early_stopping_patience"])}
        row.update(validation_metrics(splits[dataset], predict(model, splits[dataset].val.X)))
        candidate_validation_rows.append(row)
candidate_validation = pd.DataFrame(candidate_validation_rows).sort_values(["batch_size", "dataset"])
if len(candidate_validation) != len(BATCH_SIZES) * len(FEFF_DATASETS) or set(candidate_validation["batch_size"]) != set(BATCH_SIZES):
    raise AssertionError("Validation output must contain one row per candidate and FEFF task")
candidate_validation.to_csv(RUN_DIR / "candidate_validation.csv", index=False)

candidate_summary = candidate_validation.groupby("batch_size", as_index=False).agg(mean_val_eta=("val_eta", "mean"), median_val_eta=("val_eta", "median"), mean_val_median_mse=("val_median_mse", "mean"), mean_val_baseline_median_mse=("val_baseline_median_mse", "mean"))
for key in ("full_data_batches_per_epoch", "balanced_batches_per_epoch", "update_budget_scale", "effective_scheduler_patience", "effective_early_stopping_patience"):
    candidate_summary[key] = candidate_summary["batch_size"].map({int(c["batch_size"]): c[key] for c in candidate_specs})
max_score = candidate_summary["mean_val_eta"].max()
best = candidate_summary[candidate_summary["mean_val_eta"] == max_score]
if len(best) != 1:
    raise ValueError(f"Global validation selection is not unique: {len(best)} candidates tie")
selected_batch_size = int(best.iloc[0]["batch_size"])
candidate_summary["selected"] = candidate_summary["batch_size"] == selected_batch_size
candidate_summary = candidate_summary.sort_values("batch_size")
candidate_summary.to_csv(RUN_DIR / "candidate_summary.csv", index=False)
display(candidate_summary)
print(f"Selected batch size by mean validation eta: {selected_batch_size}")


,batch_size,mean_val_eta,median_val_eta,mean_val_median_mse,mean_val_baseline_median_mse,full_data_batches_per_epoch,balanced_batches_per_epoch,update_budget_scale,effective_scheduler_patience,effective_early_stopping_patience,selected
0,32,16.622119,14.893770,0.002612,0.038438,1718,614,2.798046,22,168,False
1,64,17.135905,15.094901,0.002574,0.038438,859,307,2.798046,22,168,False
2,128,17.247383,15.100621,0.002568,0.038438,430,153,2.810458,22,169,True
3,192,17.106270,14.811273,0.002579,0.038438,287,102,2.813725,23,169,False


Selected batch size by mean validation eta: 128


## 8. All-candidate test evaluation and completion artifacts

Validation selects the candidate. After selection, test metrics are computed for all four completed candidates for reporting only. The final cell prints the candidate summary, test summary, all-candidate test results, and selected-test compatibility table.


In [8]:
selected_checkpoint = candidate_checkpoints[selected_batch_size]
if set(candidate_checkpoints) != set(BATCH_SIZES):
    raise AssertionError("Test evaluation requires all four completed batch-size candidates")
test_results_all_rows = []
for batch_size, checkpoint in sorted(candidate_checkpoints.items()):
    model = load_gelu_model(checkpoint, UNIVERSAL_DROPOUT)
    for dataset in FEFF_DATASETS:
        split = splits[dataset]
        predictions = predict(model, split.test.X)
        target = split.test.y
        train_mean = split.train.y.mean(axis=0, keepdims=True)
        model_mse = float(np.median(np.mean((target - predictions) ** 2, axis=1)))
        baseline_mse = float(np.median(np.mean((target - train_mean) ** 2, axis=1)))
        if model_mse <= 0 or baseline_mse <= 0:
            raise ValueError(f"Invalid test MSE for {dataset}: model={model_mse}, baseline={baseline_mse}")
        test_results_all_rows.append({"batch_size": int(batch_size), "dataset": dataset, "element": dataset.removesuffix("_FEFF"), "checkpoint": str(checkpoint), "checkpoint_sha256": sha256_file(checkpoint), "test_median_mse": model_mse, "test_baseline_median_mse": baseline_mse, "test_eta": baseline_mse / model_mse})
test_results_all_candidates = pd.DataFrame(test_results_all_rows).sort_values(["batch_size", "dataset"])
expected_test_rows = len(candidate_checkpoints) * len(FEFF_DATASETS)
if len(test_results_all_candidates) != expected_test_rows or set(test_results_all_candidates["batch_size"]) != set(BATCH_SIZES):
    raise AssertionError("Test output must contain one row per completed candidate and FEFF task")
test_results_all_candidates.to_csv(RUN_DIR / "test_results_all_candidates.csv", index=False)

test_summary = test_results_all_candidates.groupby("batch_size", as_index=False).agg(mean_test_eta=("test_eta", "mean"), median_test_eta=("test_eta", "median"), mean_test_median_mse=("test_median_mse", "mean"), mean_test_baseline_median_mse=("test_baseline_median_mse", "mean"))
test_summary["selected"] = test_summary["batch_size"] == selected_batch_size
test_summary = test_summary.sort_values("batch_size")
if len(test_summary) != len(BATCH_SIZES):
    raise AssertionError("Test summary must contain one row per batch-size candidate")
test_summary.to_csv(RUN_DIR / "test_summary.csv", index=False)
selected_test = test_results_all_candidates[test_results_all_candidates["batch_size"] == selected_batch_size].copy()
if len(selected_test) != len(FEFF_DATASETS):
    raise AssertionError("Selected test output must contain one row per FEFF task")
selected_test.to_csv(RUN_DIR / "selected_test.csv", index=False)

manifest.update({"status": "complete", "selected_batch_size": selected_batch_size, "selected_checkpoint": str(selected_checkpoint), "selected_checkpoint_sha256": sha256_file(selected_checkpoint), "candidate_checkpoints": [{"batch_size": int(c["batch_size"]), "checkpoint": str(candidate_checkpoints[int(c["batch_size"])]), "checkpoint_sha256": sha256_file(candidate_checkpoints[int(c["batch_size"])]), "settings": s} for c, s in zip(candidate_specs, candidate_settings)], "evaluation_files": {"candidate_validation": str(RUN_DIR / "candidate_validation.csv"), "candidate_summary": str(RUN_DIR / "candidate_summary.csv"), "test_results_all_candidates": str(RUN_DIR / "test_results_all_candidates.csv"), "test_summary": str(RUN_DIR / "test_summary.csv"), "selected_test": str(RUN_DIR / "selected_test.csv")}, "selection_policy": settings_core["selection_policy"], "test_evaluation_scope": "all completed candidates for reporting after validation selection", "checkpoint_hashes_recorded": True})
manifest_path.write_text(json.dumps(json_value(manifest), indent=2), encoding="utf-8")
run_complete = {"status": "complete", "completion_label": "sweep_balanced_gelu_universal_batch_sizes_complete", "settings_manifest": str(manifest_path), "selected_batch_size": selected_batch_size, "selected_checkpoint": str(selected_checkpoint), "selected_checkpoint_sha256": sha256_file(selected_checkpoint), "selection_policy": settings_core["selection_policy"], "test_evaluation_scope": "all completed candidates for reporting after validation selection", "test_evaluated_all_candidates": True, "test_evaluated_only_for_selected_candidate": False, "evaluation_files": manifest["evaluation_files"]}
(RUN_DIR / "RUN_COMPLETE.json").write_text(json.dumps(json_value(run_complete), indent=2), encoding="utf-8")
print("\n=== Candidate summary ===")
display(candidate_summary)
print("\n=== Test summary ===")
display(test_summary)
print("\n=== All-candidate test results ===")
print(test_results_all_candidates.to_string(index=False))
print(f"\nSelected batch size: {selected_batch_size}")
print(f"Selected checkpoint: {selected_checkpoint}")
print(f"Selected checkpoint SHA256: {sha256_file(selected_checkpoint)}")



=== Candidate summary ===


,batch_size,mean_val_eta,median_val_eta,mean_val_median_mse,mean_val_baseline_median_mse,full_data_batches_per_epoch,balanced_batches_per_epoch,update_budget_scale,effective_scheduler_patience,effective_early_stopping_patience,selected
0,32,16.622119,14.893770,0.002612,0.038438,1718,614,2.798046,22,168,False
1,64,17.135905,15.094901,0.002574,0.038438,859,307,2.798046,22,168,False
2,128,17.247383,15.100621,0.002568,0.038438,430,153,2.810458,22,169,True
3,192,17.106270,14.811273,0.002579,0.038438,287,102,2.813725,23,169,False



=== Test summary ===


,batch_size,mean_test_eta,median_test_eta,mean_test_median_mse,mean_test_baseline_median_mse,selected
0,32,16.387781,14.903855,0.002521,0.037263,False
1,64,16.990392,15.552995,0.002481,0.037263,False
2,128,17.151453,15.359386,0.002457,0.037263,True
3,192,17.006250,15.091592,0.002488,0.037263,False



=== All-candidate test results ===
 batch_size dataset element                                                                                                                                                                                                    checkpoint                                                checkpoint_sha256  test_median_mse  test_baseline_median_mse  test_eta
         32 Co_FEFF      Co   /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/sweep_balanced_gelu_universal_batch_sizes/batch_size_32/best-batch32-epoch=743-val_median_mse=0.00214526.ckpt 7845ec63bb6b938aae85b2e649c12e401505417a3267c21fe8aa767a28b2319c         0.001104                  0.029960 27.133866
         32 Cr_FEFF      Cr   /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/sweep_balanced_gelu_universal_batch_sizes/batch_size_32/best-batch32-epoch=743-val_median_mse=0.00214526.ckpt 7845ec63bb6b938aae85b2e649